# Compare new IntrinsicZernikes calib tables vs old v1 maps

Compares the generated OCS map and one detector's CCS table against a
previously frozen MIW map
(`calibration/miw/intrinsic_split_maps_v1.parquet`). The new tables are
interpolated onto the old map's grid and differenced; per-Zernike in-disk
RMS of the difference is tabulated for OCS and CCS.

Note: the new CCS adds the per-CCD height **Z4 piston**, so a non-zero
Z4 CCS difference vs v1 is expected (it equals that detector's piston).
Pure numpy/scipy/matplotlib/astropy.


## 1. Parameters

In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from scipy.interpolate import LinearNDInterpolator

new_tables_dir = "/sdf/group/rubin/repo/aos_imsim/gmegias/intrinsic_zernikes/v2"
detector = 90                                   # CCS table to compare
old_maps_path = "../calibration/miw/intrinsic_split_maps_v1.parquet"

zernikes = "all"
fp_radius_deg = 1.75
r_lim = (0.1, 1.6)
pct = 98

## 2. Read both products

In [ ]:
def read_table(path):
    t = Table.read(str(path), format="parquet")
    x = t["x"].to("deg").value; y = t["y"].to("deg").value
    js = sorted(int(m.group(1)) for c in t.colnames
                for m in [re.match(r"Z(\d+)$", c)] if m)
    return x, y, js, {j: t[f"Z{j}"].to("um").value for j in js}, dict(t.meta)

xo_new, yo_new, js_o, new_ocs, _ = read_table(
    Path(new_tables_dir) / "intrinsic_aberrations_OCS.parquet")
xc_new, yc_new, js_c, new_ccs, ccs_meta = read_table(
    Path(new_tables_dir) / f"intrinsic_aberrations_CCS_det{detector:03d}.parquet")

old = Table.read(old_maps_path, format="parquet")
thx = np.asarray(old["thx_deg"], float); thy = np.asarray(old["thy_deg"], float)
js_old = sorted({int(m.group(1)) for c in old.colnames
                 for m in [re.match(r"Z(\d+)_OCS$", c)] if m})
js = sorted(set(js_o) & set(js_c) & set(js_old))
if zernikes != "all":
    js = [j for j in js if j in zernikes]
print(f"new OCS {len(xo_new)} pts; new CCS(det {detector}) {len(xc_new)} pts; "
      f"piston_z4={ccs_meta.get('piston_z4_um')} um; old {len(thx)} pts")
print(f"comparing Zernikes: {js}")

## 3. Helpers

In [ ]:
r_old = np.hypot(thx, thy)
in_disk = (r_old >= r_lim[0]) & (r_old <= r_lim[1])


def interp_to_old(x, y, vals):
    fin = np.isfinite(vals)
    f = LinearNDInterpolator(np.column_stack([x[fin], y[fin]]), np.asarray(vals)[fin])
    return f(np.column_stack([thx, thy]))


def rms_diff(a, b):
    d = (np.asarray(a) - np.asarray(b))[in_disk]; d = d[np.isfinite(d)]
    return float(np.sqrt(np.mean(d**2))) if d.size else np.nan


def plot_map(ax, vals, title, vlim, cmap="RdBu_r"):
    v = np.asarray(vals, float); fin = np.isfinite(v)
    tcf = ax.tricontourf(thx[fin], thy[fin], v[fin],
                         levels=np.linspace(-vlim, vlim, 21), cmap=cmap, extend="both")
    ax.add_patch(plt.Circle((0, 0), fp_radius_deg, fill=False, ec="k", lw=0.6, alpha=0.4))
    ax.set_aspect("equal")
    ax.set_xlim(-fp_radius_deg, fp_radius_deg); ax.set_ylim(-fp_radius_deg, fp_radius_deg)
    ax.set_title(title, fontsize=9); ax.set_xlabel("thx [deg]"); ax.set_ylabel("thy [deg]")
    return tcf


def vlim_for(*arrays):
    vv = np.concatenate([np.asarray(a, float)[np.isfinite(a)] for a in arrays])
    return max(float(np.nanpercentile(np.abs(vv), pct)), 1e-6) if vv.size else 1.0

## 4. Per-Zernike RMS-of-difference (OCS and CCS)

In [ ]:
rows = []
new_on_old = {"OCS": {}, "CCS": {}}
for j in js:
    o_new = interp_to_old(xo_new, yo_new, new_ocs[j])
    c_new = interp_to_old(xc_new, yc_new, new_ccs[j])
    new_on_old["OCS"][j] = o_new; new_on_old["CCS"][j] = c_new
    rows.append((f"Z{j}", j,
                 rms_diff(o_new, np.asarray(old[f"Z{j}_OCS"], float)),
                 rms_diff(c_new, np.asarray(old[f"Z{j}_CCS"], float))))
summary = Table(rows=rows, names=("Zernike", "j", "OCS_dRMS_um", "CCS_dRMS_um"))
for c in ("OCS_dRMS_um", "CCS_dRMS_um"):
    summary[c].format = "%.3e"
summary.pprint(max_lines=-1)
print(f"\n(CCS Z4 dRMS should be ~|piston_z4| = "
      f"{abs(ccs_meta.get('piston_z4_um', 0.0)):.4f} um)")

## 5. Per-Zernike map comparison
old / new / (new\u2212old) for each system. First two share a scale; the difference panel is self-scaled.

In [ ]:
for system, suffix in (("OCS", "_OCS"), ("CCS", "_CCS")):
    for j in js:
        old_v = np.asarray(old[f"Z{j}{suffix}"], float)
        new_v = new_on_old[system][j]
        diff = new_v - old_v
        vlim = vlim_for(old_v, new_v); dlim = vlim_for(diff)
        fig, axs = plt.subplots(1, 3, figsize=(15.5, 4.4), layout="constrained")
        t0 = plot_map(axs[0], old_v, f"Z{j} {system} old", vlim)
        plot_map(axs[1], new_v, f"Z{j} {system} new", vlim)
        fig.colorbar(t0, ax=axs[:2], shrink=0.8, label="\u00b5m")
        td = plot_map(axs[2], diff, f"Z{j} {system} new\u2212old", dlim, cmap="PuOr_r")
        fig.colorbar(td, ax=axs[2], shrink=0.8, label="\u00b5m")
        plt.show()
        plt.close(fig)